In [ ]:
import requests
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException

### 1. Ga naar website Creates

In [ ]:
# 0 Chrome opties instellen
chrome_options = Options()
chrome_options.add_argument("--incognito")  # Incognito-modus
chrome_options.add_argument("--start-maximized")  # Optioneel: scherm maximaliseren

# 1. Start de Chrome-browser
driver = webdriver.Chrome(options=chrome_options)

# 2. Ga naar de Creates website
driver.get("https://www.creates.nl/")

### 2. Ga naar AH.nl en accepteer cookies

In [ ]:
def click_cookie_button(driver):
    try:
        # Wacht tot de span met tekst "Accepteren" klikbaar is
        cookie_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//span[text()='Accepteren']"))
        )
        cookie_button.click()
        print("Cookies geaccepteerd!")
    except:
        print("Cookie-popup niet gevonden of al geaccepteerd.")
        return

In [ ]:
# 0 Chrome opties instellen
chrome_options = Options()
chrome_options.add_argument("--incognito")  # Incognito-modus
chrome_options.add_argument("--start-maximized")  # Optioneel: scherm maximaliseren

# 1. Start de Chrome-browser
driver = webdriver.Chrome(options=chrome_options)

# 2. Ga naar de Albert Heijn website
driver.get("https://www.ah.nl/")

# 3.accept cookies
click_cookie_button(driver)

# 6. Sluit de browser
driver.quit()

### 3. Ga naar pagina van winkel en scrape titel, adres en telefoonnummer van één winkel


In [ ]:
# 0. Chrome opties instellen
chrome_options = Options()
chrome_options.add_argument("--incognito")  # Incognito-modus
chrome_options.add_argument("--start-maximized")  # Optioneel: scherm maximaliseren

# 1. Start de Chrome-browser
driver = webdriver.Chrome(options=chrome_options)

# 2. Ga naar de Albert Heijn website
driver.get("https://www.ah.nl/winkel/5707")

# 3. Wacht even zodat de pagina volledig laadt (of gebruik expliciete waits)
wait = WebDriverWait(driver, 10)

# 4. Accept cookies als dat nodig is
click_cookie_button(driver)  

# 5. Vind de winkel titel met WebDriverWait
winkel_title_element = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "div.winkel_title__FyFHX h1[data-testhook='store-title']"))
)
winkel_title = winkel_title_element.text

# Vind het adres
adres_element = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "div.address-contact-details_addressColumn__SBuG8"))
)
adres_lines = adres_element.find_elements(By.TAG_NAME, "p")
adres = ", ".join([line.text for line in adres_lines])

# Vind het telefoonnummer
telefoon_element = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-testhook='store-phone'] p"))
)
telefoon = telefoon_element.text

# Print resultaten
print("Winkel titel:", winkel_title)
print("Adres:", adres)
print("Telefoonnummer:", telefoon)

# 6. Sluit de browser
driver.quit()

### 4. Scrape informatie van alle winkels en sla op in dataframe

In [ ]:
# ---- CONSTANTE SETUP ----
chrome_options = Options()
chrome_options.add_argument("--incognito")
chrome_options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 2)

# Open AH-winkelpagina en accepteer cookies (eenmalig)
driver.get("https://www.ah.nl/winkel/5707")
click_cookie_button(driver)

winkels = list()

# ---- LOOP OVER WINKELS ----
for winkel_nummer in range(1001, 10000):  # 1001 t/m 9999
    url = f"https://www.ah.nl/winkel/{winkel_nummer}"
    driver.get(url)
    
    try:
        # Winkel titel
        winkel_title_element = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.winkel_title__FyFHX h1[data-testhook='store-title']"))
        )
        winkel_title = winkel_title_element.text

        # Adres
        adres_element = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.address-contact-details_addressColumn__SBuG8"))
        )
        adres_lines = adres_element.find_elements(By.TAG_NAME, "p")
        adres = ", ".join([line.text for line in adres_lines])

        # Telefoon
        telefoon_element = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-testhook='store-phone'] p"))
        )
        telefoon = telefoon_element.text

        print(f"Winkel {winkel_nummer}:")
        print("  Titel:", winkel_title)
        print("  Adres:", adres)
        print("  Telefoon:", telefoon)

        winkels.append({
            "Winkelnummer": winkel_nummer,
            "Winkelnaam": winkel_title,
            "Adres": adres,
            "Telefoonnummer": telefoon 
        })

    except (TimeoutException, NoSuchElementException):
        print(f"Winkel {winkel_nummer} bestaat niet of kan niet geladen worden.")
        continue  # Ga naar volgende winkel

# Sluit de browser
driver.quit()

# Sla resultaten op en schrijf weg naar een CSV file
pd.DataFrame(winkels).to_csv("winkels.csv", sep=";", index=False)